In [3]:
# Step 1: Clone and build pyoqs (this takes ~5 minutes)
!apt-get install -y cmake gcc libssl-dev
!git clone --recursive https://github.com/open-quantum-safe/liboqs-python.git
%cd liboqs-python
!python3 setup.py build
!python3 setup.py install

# Step 2: Now use pyoqs for Kyber512 key generation & encapsulation
import oqs

# Generate a Kyber512 keypair
with oqs.KeyEncapsulation('Kyber512') as kem:
    public_key = kem.generate_keypair()

    # Simulate sender encapsulating a shared secret using public key
    ciphertext, shared_secret_sender = kem.encap_secret(public_key)

    # Recipient decapsulates to retrieve the same shared secret
    shared_secret_recipient = kem.decap_secret(ciphertext)

    print("Sender's Shared Secret:   ", shared_secret_sender.hex())
    print("Recipient's Shared Secret:", shared_secret_recipient.hex())

    if shared_secret_sender == shared_secret_recipient:
        print("Success! Shared secrets match.")
    else:
        print("Failure! Shared secrets do not match.")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
libssl-dev is already the newest version (3.0.2-0ubuntu1.19).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
Cloning into 'liboqs-python'...
remote: Enumerating objects: 739, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 739 (delta 198), reused 160 (delta 154), pack-reused 492 (from 2)
Receiving objects: 100% (739/739), 184.13 KiB | 2.39 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/liboqs-python/liboqs-python
python3: can't open file '/content/liboqs-python/liboqs-python/setup.py': [Errno 2] No such file or directory
python3: can't open file '/content/liboqs-python/liboqs-python/setup.py': [Errno 2] No such file or directory
Sender's Shared Secret:    fbbb768fe821cd6b24cc

In [ ]:
# Step 2: Use Dilithium from pyoqs
import oqs

# The message to sign
message = b"Post-quantum secure!"

# Generate a Dilithium key pair
with oqs.Signature('Dilithium2') as signer:
    public_key = signer.generate_keypair()
    private_key = signer.export_secret_key()

    # Sign the message
    signature = signer.sign(message)
    print("Message signed successfully!")

    # Verify the signature with the public key
    verified = signer.verify(message, signature, public_key)
    print("Signature valid on original message:", verified)

    # Modify the message
    tampered_message = b"Post-quantum is cool!"

    # Verify the signature on the tampered message (should fail)
    verified_tampered = signer.verify(tampered_message, signature, public_key)
    print("Signature valid on tampered message:", verified_tampered)


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
libssl-dev is already the newest version (3.0.2-0ubuntu1.19).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
Cloning into 'liboqs-python'...
remote: Enumerating objects: 739, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 739 (delta 198), reused 161 (delta 155), pack-reused 492 (from 2)
Receiving objects: 100% (739/739), 184.19 KiB | 1.96 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/liboqs-python/liboqs-python/liboqs-python
python3: can't open file '/content/liboqs-python/liboqs-python/liboqs-python/setup.py': [Errno 2] No such file or directory
python3: can't open file '/content/liboqs-python/liboqs-python/liboqs-python/setup.py': [Errno 2] No such file or directory
Messa

In [ ]:
# Step 2: Import libraries
import oqs
import os
import hashlib
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# Step 3: Generate Kyber512 keypair
with oqs.KeyEncapsulation('Kyber512') as kem:

    # Recipient generates Kyber keypair
    public_key = kem.generate_keypair()

    # Sender encapsulates a shared secret
    ciphertext, shared_secret_sender = kem.encap_secret(public_key)

    # Derive AES key (256-bit) from shared secret using SHA-256
    aes_key_sender = hashlib.sha256(shared_secret_sender).digest()

    # Encrypt message using AES-GCM
    aesgcm = AESGCM(aes_key_sender)
    nonce = os.urandom(12)  # AES-GCM standard nonce size
    plaintext = b"Confidential post-quantum message."
    ciphertext_encrypted = aesgcm.encrypt(nonce, plaintext, None)

    print("Sender encrypted message.")

    # Receiver decapsulates shared secret from ciphertext
    shared_secret_receiver = kem.decap_secret(ciphertext)
    aes_key_receiver = hashlib.sha256(shared_secret_receiver).digest()

    # Decrypt message using AES-GCM
    aesgcm_receiver = AESGCM(aes_key_receiver)

    try:
        decrypted = aesgcm_receiver.decrypt(nonce, ciphertext_encrypted, None)
        print("Decrypted message:", decrypted.decode())
        print("Integrity verified (authentication tag is valid).")
    except Exception as e:
        print("Decryption failed or integrity check failed:", str(e))


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
libssl-dev is already the newest version (3.0.2-0ubuntu1.19).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
Cloning into 'liboqs-python'...
remote: Enumerating objects: 739, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 739 (delta 198), reused 161 (delta 155), pack-reused 492 (from 2)
Receiving objects: 100% (739/739), 184.19 KiB | 2.30 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/liboqs-python/liboqs-python/liboqs-python/liboqs-python
python3: can't open file '/content/liboqs-python/liboqs-python/liboqs-python/liboqs-python/setup.py': [Errno 2] No such file or directory
python3: can't open file '/content/liboqs-python/liboqs-python/liboqs-python/liboqs-python/setup.py':

In [ ]:
# Step 2: Python code for tampering test
import oqs
import os
import hashlib
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

print("\nStarting key encapsulation and encryption process...")

with oqs.KeyEncapsulation('Kyber512') as kem:
    # Generate Kyber key pair
    public_key = kem.generate_keypair()

    # Sender encapsulates secret
    kyber_ciphertext, shared_secret_sender = kem.encap_secret(public_key)
    aes_key_sender = hashlib.sha256(shared_secret_sender).digest()

    # AES-GCM encrypt
    aesgcm = AESGCM(aes_key_sender)
    nonce = os.urandom(12)
    message = b"Secure message with integrity."
    ciphertext = aesgcm.encrypt(nonce, message, None)

    print("Message encrypted with AES-GCM.")

    # ------------------------------------
    # 1️⃣ Test tampering with AES ciphertext
    # ------------------------------------
    tampered_ciphertext = bytearray(ciphertext)
    tampered_ciphertext[5] ^= 0xFF  # Flip a bit
    tampered_ciphertext = bytes(tampered_ciphertext)

    print("\n Testing tampered AES-GCM ciphertext...")

    try:
        # Decapsulate normally
        shared_secret_receiver = kem.decap_secret(kyber_ciphertext)
        aes_key_receiver = hashlib.sha256(shared_secret_receiver).digest()

        # Attempt decryption
        aesgcm_receiver = AESGCM(aes_key_receiver)
        decrypted = aesgcm_receiver.decrypt(nonce, tampered_ciphertext, None)
        print("Unexpected success: Decrypted message:", decrypted.decode())
    except Exception as e:
        print("AES-GCM detected tampering. Decryption failed:", str(e))

    # ------------------------------------
    #  Test tampering with Kyber ciphertext
    # ------------------------------------
    print("\nTesting tampered Kyber ciphertext...")

    tampered_kem_ct = bytearray(kyber_ciphertext)
    tampered_kem_ct[0] ^= 0xAA  # Flip a bit
    tampered_kem_ct = bytes(tampered_kem_ct)

    try:
        # Attempt decapsulation with tampered ciphertext
        shared_secret_tampered = kem.decap_secret(tampered_kem_ct)
        aes_key_fake = hashlib.sha256(shared_secret_tampered).digest()

        # Try decrypting original ciphertext with incorrect key
        aesgcm_fake = AESGCM(aes_key_fake)
        decrypted = aesgcm_fake.decrypt(nonce, ciphertext, None)
        print("Unexpected success: Decrypted message:", decrypted.decode())
    except Exception as e:
        print("Kyber tampering detected. Decryption or tag check failed:", str(e))


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
libssl-dev is already the newest version (3.0.2-0ubuntu1.19).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
Cloning into 'liboqs-python'...
remote: Enumerating objects: 739, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 739 (delta 198), reused 161 (delta 155), pack-reused 492 (from 2)
Receiving objects: 100% (739/739), 184.19 KiB | 2.19 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python
python3: can't open file '/content/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python/setup.py': [Errno 2] No such file or directory
python3: can't open file '/content/liboqs-pyt

In [ ]:
# Step 2: Timing with time module
import oqs
import time


with oqs.KeyEncapsulation('Kyber512') as kem:
    # Key generation
    t1 = time.time()
    public_key = kem.generate_keypair()
    t2 = time.time()
    print(f"Kyber Key Generation: {t2 - t1:.6f} seconds")

    # Encapsulation
    t1 = time.time()
    ciphertext, shared_secret = kem.encap_secret(public_key)
    t2 = time.time()
    print(f"Kyber Encapsulation: {t2 - t1:.6f} seconds")

    # Decapsulation
    t1 = time.time()
    shared_secret_recv = kem.decap_secret(ciphertext)
    t2 = time.time()
    print(f"Kyber Decapsulation: {t2 - t1:.6f} seconds")


print("\n⏱ Measuring Dilithium2 Signature Timings")

with oqs.Signature('Dilithium2') as sig:
    # Key generation
    t1 = time.time()
    public_key = sig.generate_keypair()
    t2 = time.time()
    print(f"Dilithium Key Generation: {t2 - t1:.6f} seconds")

    # Sign
    message = b"Post-quantum secure!"
    t1 = time.time()
    signature = sig.sign(message)
    t2 = time.time()
    print(f"Dilithium Signing: {t2 - t1:.6f} seconds")

    # Verify
    t1 = time.time()
    valid = sig.verify(message, signature, public_key)
    t2 = time.time()
    print(f"Dilithium Verification: {t2 - t1:.6f} seconds")
    print(f"Signature valid? {valid}")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
libssl-dev is already the newest version (3.0.2-0ubuntu1.19).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
Cloning into 'liboqs-python'...
remote: Enumerating objects: 739, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 739 (delta 198), reused 160 (delta 154), pack-reused 492 (from 2)
Receiving objects: 100% (739/739), 184.13 KiB | 2.33 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python
python3: can't open file '/content/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python/liboqs-python/setup.py': [Errno 2] No such file or directory
python3: can't op